In [1]:
import pandas as pd
import os
import re
from lamma_ask import run_llama

df= pd.read_csv('LifeQA/for_eval/new/OIC_llama_wo_st_updated.csv')
df['subtitle']= df['subtitle'].fillna('No subtitle')

df = df.drop_duplicates()
df.reindex()
df = df.dropna()
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df

,q_id,question,answer_type,Answers,video_id,start,end,subtitle,OIC_context,OIC_question,ans_idx,OIC_answer_llama,Match_llama
0,1011,How many people are in the car?,V,"['11', '4', '2', '7']",TGUYv10XdTI,19.328000,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,How many people are in the car?Guess the most ...,2,2,Correct
1,1012,What color is the man's hat?,V,"['silver', 'golden', 'blue', 'red']",TGUYv10XdTI,19.328000,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,What color is the man's hat?Guess the most lik...,2,1,Wrong
2,1013,Where are they?,B,"['restroom', 'train', 'bedroom', 'car']",TGUYv10XdTI,19.328000,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where are they?Guess the most likely answer am...,3,2,Wrong
3,1014,Where are they going?,L,"['class', 'restroom', 'party', 'mall']",TGUYv10XdTI,19.328000,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where are they going?Guess the most likely ans...,3,2,Wrong
4,1015,Where does she want to go to first?,L,"['Aldo', 'home', 'Zara', 'Forever XXI']",TGUYv10XdTI,19.328000,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where does she want to go to first?Guess the m...,3,2,Wrong
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1911,2282,How many full glasses of water are?,B,"['1', '2', '3', '13']",SrBp1Ojt5Z0,136.533333,225.664000,[woman] ok so James do you think these both gl...,The scene opens with glass_e8e6 on table_8713\...,How many full glasses of water are?Guess the m...,1,2,Wrong
1912,2283,What is the boy's name?,L,"['Joel', 'James', 'Mario', 'Jean']",SrBp1Ojt5Z0,136.533333,225.664000,[woman] ok so James do you think these both gl...,The scene opens with glass_e8e6 on table_8713\...,What is the boy's name?Guess the most likely a...,1,2,Wrong
1913,2284,How many cups are there?,V,"['3', '1', '4', '2']",SrBp1Ojt5Z0,136.533333,225.664000,[woman] ok so James do you think these both gl...,The scene opens with glass_e8e6 on table_8713\...,How many cups are there?Guess the most likely ...,0,1,Wrong
1914,2285,Does James think that there is the same amount...,L,"['yes', '', 'no', '']",SrBp1Ojt5Z0,136.533333,225.664000,[woman] ok so James do you think these both gl...,The scene opens with glass_e8e6 on table_8713\...,Does James think that there is the same amount...,0,1,Wrong


In [2]:
df['OIC_context_without_CUID'] =  df['OIC_context'].apply(lambda x: re.sub('_[a-zA-Z0-9]*', '', x))
df['OIC_context_without_CUID'][0]

'The scene opens with woman wearing shirt\nwoman in car\nwoman wearing hat\nAfter 1 seconds:\near of woman\nAfter 2 seconds:\nshirt on woman\nAfter     7 seconds:\nhat on head\nAfter  9 seconds:\nshirt on woman\nwoman wearing shirt\nAfter  11 seconds:\nshirt on woman\nhat on head\nhat on woman\nAfter 12 seconds:\near of man\nman wearing shirt\nman wearing hat\nAfter  14 seconds:\nman in car\nAfter 15 seconds:\nshirt on woman\near of woman\nwoman wearing hat\nAfter 16 seconds:\nwoman wearing hat\nboy wearing shirt\nglass on face\nAfter 17 seconds:\near of boy\nboy wearing hat\nAfter  19 seconds:\nshirt on woman\near of woman\nwoman wearing shirt\nwoman wearing hat\nboy wearing hat\nAfter 20 seconds:\nwoman wearing glass\nAfter 21 seconds:\nwoman wearing hat\nAfter 22 seconds:\nwoman in car\nwoman wearing glass\nwoman in car\nAfter 23 seconds:\nwoman holding phone\nwoman wearing tie\nhat on woman\nAfter  25 seconds:\nphone in hand\nAfter 26 seconds:\nhat on woman\nboy wearing shirt\nboy 

In [3]:
class Substitutable(str):
  def __new__(cls, *args, **kwargs):
    newobj = str.__new__(cls, *args, **kwargs)
    newobj.sub = lambda fro,to: Substitutable(re.sub(fro, to, newobj))
    return newobj

In [4]:
df['OIC_context_without_CUID_temp'] = df['OIC_context_without_CUID'].apply(lambda x: Substitutable(x).sub('\n', ', ').sub('The scene opens with', '').sub('After', '').sub(' [0-9]* seconds:', ''))

df['OIC_context_without_CUID_temp'][0]

' woman wearing shirt, woman in car, woman wearing hat, , ear of woman, , shirt on woman,     , hat on head,  , shirt on woman, woman wearing shirt,  , shirt on woman, hat on head, hat on woman, , ear of man, man wearing shirt, man wearing hat,  , man in car, , shirt on woman, ear of woman, woman wearing hat, , woman wearing hat, boy wearing shirt, glass on face, , ear of boy, boy wearing hat,  , shirt on woman, ear of woman, woman wearing shirt, woman wearing hat, boy wearing hat, , woman wearing glass, , woman wearing hat, , woman in car, woman wearing glass, woman in car, , woman holding phone, woman wearing tie, hat on woman,  , phone in hand, , hat on woman, boy wearing shirt, boy wearing hat, , shirt on woman, ear of woman, , shirt on woman, woman wearing hat,  , woman in car, woman wearing hat, woman wearing hat,   , woman in car,  , woman wearing hat, hat on woman, , woman wearing hat, , hair on woman, woman in car, , shirt on woman, , hair on woman, window on car, , nose on fa

In [5]:
df.head()

,q_id,question,answer_type,Answers,video_id,start,end,subtitle,OIC_context,OIC_question,ans_idx,OIC_answer_llama,Match_llama,OIC_context_without_CUID,OIC_context_without_CUID_temp
0,1011,How many people are in the car?,V,"['11', '4', '2', '7']",TGUYv10XdTI,19.328,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,How many people are in the car?Guess the most ...,2,2,Correct,The scene opens with woman wearing shirt\nwoma...,"woman wearing shirt, woman in car, woman wear..."
1,1012,What color is the man's hat?,V,"['silver', 'golden', 'blue', 'red']",TGUYv10XdTI,19.328,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,What color is the man's hat?Guess the most lik...,2,1,Wrong,The scene opens with woman wearing shirt\nwoma...,"woman wearing shirt, woman in car, woman wear..."
2,1013,Where are they?,B,"['restroom', 'train', 'bedroom', 'car']",TGUYv10XdTI,19.328,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where are they?Guess the most likely answer am...,3,2,Wrong,The scene opens with woman wearing shirt\nwoma...,"woman wearing shirt, woman in car, woman wear..."
3,1014,Where are they going?,L,"['class', 'restroom', 'party', 'mall']",TGUYv10XdTI,19.328,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where are they going?Guess the most likely ans...,3,2,Wrong,The scene opens with woman wearing shirt\nwoma...,"woman wearing shirt, woman in car, woman wear..."
4,1015,Where does she want to go to first?,L,"['Aldo', 'home', 'Zara', 'Forever XXI']",TGUYv10XdTI,19.328,76.650667,[boy] hello [girl] hey\n[boy] where are we goi...,The scene opens with woman_766a wearing shirt_...,Where does she want to go to first?Guess the m...,3,2,Wrong,The scene opens with woman wearing shirt\nwoma...,"woman wearing shirt, woman in car, woman wear..."


In [6]:
main=df
import ast
q_ids = main['q_id'].unique()
i=0
for qid in q_ids:
  #if 'Seq' in q:
    que = main.query("q_id=={}".format(qid))
    question = que['question'].values[0]
    answer_id = que["ans_idx"].values[0]
    subtitle = que['subtitle'].values[0]
    context = que['OIC_context_without_CUID_temp'].values[0]
    #print(context)
    
    prompt = context
    
    
    options = ast.literal_eval(que['Answers'].values[0])
    choice_string = ''

    choice_string = "0: {}, 1: {}, 2: {}, 3: {}".format(options[0], options[1], options[2], options[3])
    
    formatted_question = question+ 'Guess the most likely answer among these options: '+choice_string+' Respond with a only a single integer between 0 and 3. Do not produce any other or verbose answers. If enough information is not given, still make a guess to result in one out of the given options. Do not explain yourself, answer with only an integer value.'
    print('-'*100)
    print(i)
    #print(formatted_question)
    #response = run_gpt(prompt+'\n subtitle: '+subtitle, formatted_question)
    #response = run_gpt(prompt, formatted_question)
    response = run_llama(prompt, formatted_question)
    main.loc[main['q_id'] == qid, 'OIC_answer_llama'] = str(response)
    main.loc[main['q_id'] == qid, 'OIC_question'] = formatted_question
    main.loc[main['q_id'] == qid, 'OIC_context'] = prompt
    OIC_answer = response
    print('OIC question: {}'.format(formatted_question))
    print(choice_string)
    print('OIC answer: {}'.format(OIC_answer))
    if len(OIC_answer)>1:
      main.loc[main['q_id'] == qid, 'Match_llama_wo_CUID_temp'] = OIC_answer
    else:
      if int(OIC_answer) == int(answer_id):
          main.loc[main['q_id'] == qid, 'Match_llama_wo_CUID_temp'] = 'Correct'
          print('correct')
      else:
          print('wrong')
          main.loc[main['q_id'] == qid, 'Match_llama_wo_CUID_temp'] = 'Wrong'
    
    main.to_csv('LifeQA/for_eval/new/OIC_llama_modified_wo_CUID_temp.csv')
    i=i+1
    

----------------------------------------------------------------------------------------------------
0
OIC question: How many people are in the car?Guess the most likely answer among these options: 0: 11, 1: 4, 2: 2, 3: 7 Respond with a only a single integer between 0 and 3. Do not produce any other or verbose answers. If enough information is not given, still make a guess to result in one out of the given options. Do not explain yourself, answer with only an integer value.
0: 11, 1: 4, 2: 2, 3: 7
OIC answer: 2
correct
----------------------------------------------------------------------------------------------------
1
OIC question: What color is the man's hat?Guess the most likely answer among these options: 0: silver, 1: golden, 2: blue, 3: red Respond with a only a single integer between 0 and 3. Do not produce any other or verbose answers. If enough information is not given, still make a guess to result in one out of the given options. Do not explain yourself, answer with only an 